In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

np.random.seed(42)

In [ ]:
data = pd.read_csv("data/train.csv").to_numpy(dtype=np.float32).copy()
np.random.shuffle(data)

test_data = data[:5000]
val_data = data[5000:10000]
train_data = data[10000:]


def prepare(data):
    y = data[:, 0].astype(int)
    X = np.ascontiguousarray((data[:, 1:] / 255).T)
    Y = np.zeros((10, y.size), dtype=np.float32)
    Y[y, np.arange(y.size)] = 1
    return X, Y, y


X_train, Y_train, y_train = prepare(train_data)
X_val, Y_val, y_val = prepare(val_data)
X_test, Y_test, y_test = prepare(test_data)

print(X_train.shape, Y_train.shape, y_train.shape)
print(X_val.shape, Y_val.shape, y_val.shape)
print(X_test.shape, Y_test.shape, y_test.shape)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[:, i].reshape(28, 28), cmap="gray")
    ax.set_title(f"label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def init_params(hidden_size):
    W1 = (np.random.randn(hidden_size, 784) * np.sqrt(2 / 784)).astype(np.float32)
    b1 = np.zeros((hidden_size, 1), dtype=np.float32)
    W2 = (np.random.randn(10, hidden_size) * np.sqrt(2 / hidden_size)).astype(np.float32)
    b2 = np.zeros((10, 1), dtype=np.float32)
    return W1, b1, W2, b2


def relu(Z):
    return np.maximum(0, Z)


def softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return expZ / np.sum(expZ, axis=0, keepdims=True)


def forward_prop(X, W1, b1, W2, b2):
    A0 = X
    Z1 = W1 @ A0 + b1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [ ]:
def compute_loss(A2, Y):
    m = Y.shape[1]
    return -np.sum(Y * np.log(A2 + 1e-8)) / m


def get_predictions(A2):
    return np.argmax(A2, axis=0)


def get_accuracy(predictions, y):
    return np.mean(predictions == y)

In [ ]:
def relu_deriv(Z):
    return Z > 0


def backward_prop(X, Y, Z1, A1, A2, W2):
    m = Y.shape[1]
    dZ2 = A2 - Y
    dW2 = (1 / m) * dZ2 @ A1.T
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = (W2.T @ dZ2) * relu_deriv(Z1)
    dW1 = (1 / m) * dZ1 @ X.T
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2

In [ ]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

In [ ]:
def gradient_check(hidden_size=16, samples=20, checks=5, epsilon=1e-5):
    X = X_train[:, :samples].astype(np.float64)
    Y = Y_train[:, :samples].astype(np.float64)
    params = [p.astype(np.float64) for p in init_params(hidden_size)]
    Z1, A1, Z2, A2 = forward_prop(X, *params)
    grads = backward_prop(X, Y, Z1, A1, A2, params[2])
    for name, param, grad in zip(["W1", "b1", "W2", "b2"], params, grads):
        worst = 0
        for _ in range(checks):
            i = np.random.randint(param.shape[0])
            j = np.random.randint(param.shape[1])
            original = param[i, j]
            param[i, j] = original + epsilon
            loss_plus = compute_loss(forward_prop(X, *params)[3], Y)
            param[i, j] = original - epsilon
            loss_minus = compute_loss(forward_prop(X, *params)[3], Y)
            param[i, j] = original
            numeric = (loss_plus - loss_minus) / (2 * epsilon)
            analytic = grad[i, j]
            difference = abs(numeric - analytic) / max(abs(numeric) + abs(analytic), 1e-12)
            worst = max(worst, difference)
        print(name, "worst relative difference:", worst)


gradient_check()

In [ ]:
def train(X, Y, y, X_val, Y_val, y_val, hidden_size, alpha, iterations):
    W1, b1, W2, b2 = init_params(hidden_size)
    hist = {"iter": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(X, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward_prop(X, Y, Z1, A1, A2, W2)

        if i % 50 == 0:
            _, _, _, A2_val = forward_prop(X_val, W1, b1, W2, b2)
            hist["iter"].append(i)
            hist["train_loss"].append(compute_loss(A2, Y))
            hist["val_loss"].append(compute_loss(A2_val, Y_val))
            hist["train_acc"].append(get_accuracy(get_predictions(A2), y))
            hist["val_acc"].append(get_accuracy(get_predictions(A2_val), y_val))
            print(f"iter {i:4d} | train loss {hist['train_loss'][-1]:.4f} acc {hist['train_acc'][-1]:.4f}"
                  f" | val loss {hist['val_loss'][-1]:.4f} acc {hist['val_acc'][-1]:.4f}")

        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

    return W1, b1, W2, b2, hist

In [ ]:
hidden_size = 256
alpha = 1.0
iterations = 2000

start = time.time()
W1, b1, W2, b2, hist = train(X_train, Y_train, y_train, X_val, Y_val, y_val,
                             hidden_size, alpha, iterations)
print(f"time: {time.time() - start:.1f} s")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(hist["iter"], hist["train_loss"], label="train")
ax1.plot(hist["iter"], hist["val_loss"], label="validation")
ax1.set_xlabel("iteration")
ax1.set_ylabel("loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(hist["iter"], hist["train_acc"], label="train")
ax2.plot(hist["iter"], hist["val_acc"], label="validation")
ax2.set_xlabel("iteration")
ax2.set_ylabel("accuracy")
ax2.set_title("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
_, _, _, A2_train = forward_prop(X_train, W1, b1, W2, b2)
_, _, _, A2_val = forward_prop(X_val, W1, b1, W2, b2)

print("train accuracy:", get_accuracy(get_predictions(A2_train), y_train))
print("validation accuracy:", get_accuracy(get_predictions(A2_val), y_val))

In [ ]:
_, _, _, A2_test = forward_prop(X_test, W1, b1, W2, b2)
test_predictions = get_predictions(A2_test)

print("test loss:", compute_loss(A2_test, Y_test))
print("test accuracy:", get_accuracy(test_predictions, y_test))
print("wrong:", np.sum(test_predictions != y_test), "out of", y_test.size)

In [ ]:
cm = np.zeros((10, 10), dtype=int)
np.add.at(cm, (y_test, test_predictions), 1)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("predicted digit")
ax.set_ylabel("true digit")
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

In [ ]:
wrong_idx = np.where(test_predictions != y_test)[0]

fig, axes = plt.subplots(3, 5, figsize=(11, 7))
for ax, idx in zip(axes.flat, wrong_idx[:15]):
    ax.imshow(X_test[:, idx].reshape(28, 28), cmap="gray")
    conf = A2_test[test_predictions[idx], idx]
    ax.set_title(f"true {y_test[idx]} | pred {test_predictions[idx]}\nconf {conf:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
np.savez("weights.npz", W1=W1, b1=b1, W2=W2, b2=b2)

In [ ]:
weights = np.load("weights.npz")
W1, b1, W2, b2 = weights["W1"], weights["b1"], weights["W2"], weights["b2"]


def predict_image(image):
    _, _, _, A2 = forward_prop(image.reshape(784, 1), W1, b1, W2, b2)
    return np.argmax(A2), A2.max()


index = 0
digit, confidence = predict_image(X_test[:, index])
print("predicted:", digit, "| confidence:", round(float(confidence), 3), "| true:", y_test[index])

plt.imshow(X_test[:, index].reshape(28, 28), cmap="gray")
plt.title(f"predicted {digit}")
plt.axis("off")
plt.show()